# Environment Setting

In [1]:
!pip install --upgrade openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.8/328.8 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.6 MB/s eta 0:00:00


In [2]:
import openai
from openai import OpenAI
import os
import pandas as pd

## Concatenate all restaurant reviews with its name

In [ ]:
import glob
import os
import pandas as pd

# List all CSV files
csv_files = glob.glob('*.csv')

# Initialize an empty list to store DataFrames
dataframes = []

# Loop through each CSV file and read it into a DataFrame
for file in csv_files:
    # Read the CSV file into a DataFrame
    df = pd.read_csv(file)

    # If the DataFrame has more than 100 rows, randomly choose 100 rows
    if len(df) > 100:
        df = df.sample(n=100)

    # Extract the restaurant name from the file name (without extension)
    restaurant_name = os.path.splitext(os.path.basename(file))[0]

    # Add a column with the restaurant name
    df['restaurant_name'] = restaurant_name

    # Ensure the restaurant name is the first column
    df = df[['restaurant_name'] + [col for col in df.columns if col != 'restaurant_name']]

    # Append the DataFrame to the list
    dataframes.append(df)

# Concatenate all DataFrames
merged_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
merged_df

,restaurant_name,Name,Profile Location,Score,Date,Elite 24,Friends,Reviews,Photos,reserved,pictures,checkin,Comment,Helpful,Thanks,Love this,Oh no,Reply Date,Reply Content
0,LoveMama,Tedd J.,"Evanston, IL",1,"Jun 17, 2024",no,24,1,15,no,NaN,NaN,"Noodle soup, smell bad & cold\nA lotttttttt of...",0,0,0,0,NaN,NaN
1,LoveMama,Mike B.,"Los Angeles, CA",5,"Jun 15, 2024",no,0,11,0,no,NaN,NaN,"Incredible food, and the staff was amazing. Lo...",0,0,0,0,NaN,NaN
2,LoveMama,Lisa M.,"Bronx, Bronx, NY",4,"Jun 15, 2024",no,0,68,4,yes,NaN,NaN,Good food but service could be better. Older w...,0,0,0,0,NaN,NaN
3,LoveMama,Grace H.,"Norco, CA",5,"Jun 11, 2024",no,141,13,6,no,NaN,NaN,Pretty amazing food . Very reasonable price . ...,0,0,0,0,NaN,NaN
4,LoveMama,Lynn P.,"Los Angeles, CA",5,"Jun 3, 2024",no,0,3,0,yes,NaN,NaN,All the food we ordered was tasty and the serv...,0,0,0,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Di Fara Pizza,Amanda A.,"Mount Vernon, WA",4,"Apr 7, 2023",no,33,167,176,no,4 photos,NaN,"Visited by famous people, the average Joe, and...",0,0,1,0,NaN,NaN
996,Di Fara Pizza,Jennifer T.,"San Francisco, CA",4,"Apr 3, 2023",no,21,246,698,no,2 photos,NaN,"We picked up two large pizzas, one with half w...",0,0,0,1,NaN,NaN
997,Di Fara Pizza,Huey N.,"Sunnyvale, CA",4,"Apr 3, 2023",no,565,668,56,no,NaN,NaN,Di Fara is indeed worth the hype and going out...,1,0,1,0,NaN,NaN
998,Di Fara Pizza,Carolina S.,"Washington, DC",3,"Mar 30, 2023",no,9,69,103,no,3 photos,NaN,Came here to try the famous Di Fara Pizza. We ...,4,1,2,0,NaN,NaN


In [ ]:
count = merged_df['restaurant_name'].value_counts()
count

restaurant_name
LoveMama                     100
Amy Ruth’s                   100
L & B Spumoni Gardens        100
Serendipity 3                100
Buddakan                     100
The Halal Guys               100
Ippudo NY                    100
Artichoke Basille’s Pizza    100
Red Hook Lobster Pound       100
Di Fara Pizza                100
Name: count, dtype: int64

# Prompt Engineering: summarize and extract

In [3]:
merged_df = pd.read_csv('nyc_100_samples_10_restaurants.csv')

In [ ]:
def get_response(prompt):
  # Create a request to the chat completions endpoint
  response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    # Assign the role and content for the message
    messages=[{"role": "user", "content": prompt}],
    temperature = 0)
  return response.choices[0].message.content

In [5]:
def extract_topics(text):
  instructions = """Extract main business process topics with maximum 4 from the following review.
                    Categorize each issue under the relevant area
                    (Food Quality, Customer Service, Cleanliness, Ambiance, Value for Money, Order Accuracy and efficiency, Waiting time, kids or pets friendly, menu choice),
                    and provide a categorization, description and attitude of satisfaction or not for each issue, no need mention topics that are not involved in the review."""
  output_format = """Structure each topic as follows:
                  - Category: <category of topics mentioned in the review>
                  - Desctiption: <description of the topics in the review in one sentence>
                  - Attitude: <sentiment of this topic in one word>"""
  prompt = instructions + output_format + f"""the review is as follows: {text}"""
  response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Use a supported chat model
        messages=[
            {"role": "system", "content": "You are a helpful assistant for analyzing online reviews for restaurants. "},
            {"role": "user", "content": prompt}
        ],
        max_tokens= 150
    )
  return response.choices[0].message.content.strip()  # Extract content from the message object

## Approach 1: Direct Entry of extraction

In [15]:
sample_df_60_100 = merged_df.loc[60:,:]

In [16]:
sample_df_60_100['topics'] = sample_df_60_100['Comment'].apply(extract_topics)

<ipython-input-16-30629c63e4ce>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df_60_100['topics'] = sample_df_60_100['Comment'].apply(extract_topics)


In [17]:
sample_df_60_100[['Comment','topics']]

,Comment,topics
60,I've gone here twice and it's a staple for Tha...,- Category: Food Quality\n - Description: The...
61,No frills Vietnamese/Thai/Malaysian restaurant...,- Category: Food Quality\n - Description: Lak...
62,The food was great. Service was fast and the s...,- Category: Food Quality\n - Description: The...
63,The food...the vibe...and grandpa\nLaksa was a...,- Category: Food Quality\n - Description: Lak...
64,Dinner took me back to having meals in Asia wh...,- Category: Food Quality\n - Description: The...
...,...,...
995,"Visited by famous people, the average Joe, and...",- Category: Food Quality\n - Description: The...
996,"We picked up two large pizzas, one with half w...",- Category: Food Quality\n - Description: The...
997,Di Fara is indeed worth the hype and going out...,- Category: Food Quality\n - Description: Rem...
998,Came here to try the famous Di Fara Pizza. We ...,- Category: Food Quality\n - Description: The...


In [13]:
sample_df_40_60['topics'] = sample_df_40_60['Comment'].apply(extract_topics)
sample_df_40_60[['Comment','topics']]

<ipython-input-13-8a1ecc0af295>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df_40_60['topics'] = sample_df_40_60['Comment'].apply(extract_topics)


,Comment,topics
40,The best pad thai I've ever had!! A true mom a...,- Category: Food Quality\n - Description: Del...
41,"Service is excellent and quick, the owners are...",- Category: Food Quality\n - Description: Tom...
42,fantastic food for very low prices. the servic...,- Category: Food Quality\n - Description: Fan...
43,"Really fast, and good! Great service, no frill...",- Category: Food Quality\n - Description: Goo...
44,"Food was great, well priced, service was amazi...",- Category: Food Quality\n - Description: Foo...
45,These niggas cook like my mom. Made me feel at...,- Category: Food Quality\n - Description: Lik...
46,"great food, affordable, nice people! one of my...",- Category: Food Quality\n - Description: Fla...
47,"Straightforward Thai, Malaysian, and Vietnames...",- Category: Food Quality\n - Description: Vie...
48,Great food! Authentic! Nice people! Good quiet...,- Category: Food Quality\n - Description: Gre...
49,"Good food, good service, good vibes. Recommend...",- Category: Food Quality\n - Description: Rev...


In [9]:
sample_df_20_40[['Comment','topics']]

,Comment,topics
20,Great food at relatively reasonable prices. On...,- Category: Food Quality\n - Description: Ste...
21,Great food that has a home cooked feel. Easy t...,- Category: Food Quality\n - Description: Gre...
22,Great food and great prices. The passionfruit ...,- Category: Food Quality\n - Description: Gre...
23,What can you say? Nasty people. Nasty food. ...,- Category: Customer Service\n - Description:...
24,Came down to this no frill southeast Asian pla...,- Category: Food Quality\n - Description: Di...
25,Just had lunch here with my husband. We tried ...,- Category: Food Quality\n - Description: Gri...
26,Good was great and service was great. Loved th...,- Category: Food Quality\n- Description: Good ...
27,My family and I initially chose this spot beca...,- Category: Food Quality\n - Description: The...
28,You can tell this restaurant has been here fo...,- Category: Food Quality\n - Description: The...
29,The food at LoveMama was so tasty and deliciou...,- Category: Food Quality\n - Description: The...


In [18]:
sample_df_60_100.to_csv('sample_df_60_1000_240722_0.csv', index=False)

In [ ]:
LA_littlesister = pd.read_csv('Little Sister_1956.csv')
LA_littlesister_sample_10 = LA_littlesister.sample(n=10)

In [ ]:
LA_littlesister_sample_10['topics'] = LA_littlesister_sample_10['Comment'].apply(extract_topics)
LA_littlesister_sample_10[['Comment','topics']]

,Comment,topics
1379,Certain items were a hit or a miss. Found some...,- Category: Food Quality\n - Description: Cer...
1824,This is a must visit if you're in the DTLA are...,- Category: Ambiance\n - Description: The amb...
1279,My husband and I had our anniversary lunch her...,- Category: Food Quality\n - Description: Rev...
577,"Like the food there, pretty delicate and authe...",- Category: Ambiance\n- Description: Narrow sp...
158,I've been wanting to try Little Sister and I w...,- Category: Food Quality\n - Description: The...
1916,This place is really yummy and surprisingly au...,- Category: Food Quality\n - Description: Dis...
900,Today's order as written on the menu:\nIntrodu...,- Category: Food Quality\n - Description: Var...
210,I had an great dining experience at Little Sis...,- Category: Ambiance\n - Description: Cozy an...
682,"In LA for the marathon, a friend recommended L...",- Category: Food Quality\n - Description: The...
1620,Just ate lunch here and the service was incred...,- Category: Food Quality\n - Description: Fre...


### Try a framework of extracting topics

In [ ]:
def framework_extract(text):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Use a supported chat model
        messages=[
            {"role": "system", "content": "You are a helpful assistant for analyzing online reviews for restaurants. Your task is to extract and identify up to 3 business process issues from the reviews. For each issue, provide a Problem Description, Impact, and Suggested Solution, each limited to 10 words or fewer."},
            {"role": "user", "content": f"Extract up to 3 business process issues from the following review. For each issue, provide a Problem Description, Impact, and Suggested Solution, each in no more than 10 words:\n\n{text}"}
        ],
        max_tokens=150
    )

    result = response.choices[0].message.content.strip()

    # Format the result as a table-friendly structure
    problems = result.split('\n\n')  # Split each issue
    structured_issues = []

    for problem in problems:
        parts = problem.split('\n')
        if len(parts) == 3:  # Ensure we have Problem Description, Impact, and Suggested Solution
            structured_issues.append({
                'Problem Description': parts[0].replace('Problem Description: ', ''),
                'Impact': parts[1].replace('Impact: ', ''),
                # 'Suggested Solution': parts[2].replace('Suggested Solution: ', '')
            })

    return structured_issues

# Example usage
sample_df = merged_df.sample(n=5)
sample_df['topics'] = sample_df['Comment'].apply(framework_extract)

In [ ]:
sample_df[['Comment','topics']]

,Comment,topics
150,3.89 Stars is my actual score. \nSo here's the...,[]
855,"Thanks to a tip from Matteo, I knew I had to g...","[{'Problem Description': '1. Slow service', 'I..."
269,Absolutely amazing. The pizza was a unique sty...,[{'Problem Description': '1. Problem: Inconsis...
985,Best pizza I have tasted and I traveled hr and...,[]
460,"This is definitely a happening place in NYC, a...",[]


# Approach 2: Extract after clarrification

## 4 types of attitudes: Good, Good but something can be improved, Neutral, Bad

In [ ]:
import pandas as pd
from textblob import TextBlob

# Load your dataset
reviews_df = pd.read_csv('nyc_100_samples_10_restaurants.csv')

# Function to classify review attitudes based on sentiment polarity
def classify_attitude(polarity):
    if polarity > 0.5:
        return 'Good'
    elif 0 < polarity <= 0.5:
        return 'Good but something can be improved'
    elif polarity == 0:
        return 'Neutral'
    else:
        return 'Bad'

# Sentiment Analysis on comments
reviews_df['sentiment'] = reviews_df['Comment'].apply(lambda x: TextBlob(x).sentiment.polarity)

# Classify attitudes
reviews_df['attitude'] = reviews_df['sentiment'].apply(classify_attitude)


In [ ]:
Summarize the problem in one sentence. If the text is about the issues, Structure the content as
                        - Problem: <the problems mentioned in the text>
                        - Topic:<the topics of the problem accordingly>
                        If the text is about positive feedback, structure the content as
                        - Positive feedback: <the positive feedback mentioned in the text>
                        - Topic:<the topics of the positive feedback accordingly>

In [ ]:
def extract_topics(text):
    # Create the instructions
    instructions = f"""You will be provided with a text delimited by triple backticks. If the text mentions some issues of the restaurant, describe the problems mentioned, then generate a topic for each problem. Otherwise, if the text involves only good reviews, generate the topics of the positive feedback."""

    # Create the output format
    output_format = f"""Transform the text delimited by triple backticks with the following two steps:
                        Step 1 - Reformat it to summarize the problems or good points mentioned with proofreading in one sentence.
                        Step 2 - Extract the topics of problems or good in a list like ['Service','food quality','Ambiance']
                        ```{text}```"""
    prompt = instructions + output_format + f"```{text}```"
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Use a supported chat model
        messages=[
            {"role": "system", "content": "You are a helpful assistant for analyzing online reviews for restaurants."},
            {"role": "user", "content": prompt}
        ],
        max_tokens= 100
    )
    return response.choices[0].message.content.strip()  # Extract content from the message object

In [ ]:
# Extract topics for each attitude category
sample_df = reviews_df.sample(n=5)
attitude_groups = sample_df.groupby('attitude')

topics_by_attitude = {}

for attitude, group in attitude_groups:
    combined_text = " ".join(group['Comment'].tolist())
    topics = extract_topics(combined_text)
    topics_by_attitude[attitude] = topics

# Display topics for each attitude category
for attitude, topics in topics_by_attitude.items():
    print(f"Topics for {attitude} reviews:")
    print(topics)
    print("\n")

Topics for Bad reviews:
Step 1: The reviewer expressed dissatisfaction with the restaurant, mentioning the issue of burned food and poor service. The customer specifically asked if the food was burned from the bottom, and the staff denied it. The reviewer's overall experience was negative, leading them to declare they will not return to the restaurant.

Step 2: 
- Burned food quality
- Poor customer service


Topics for Good reviews:
**Step 1:** The restaurant has been established for a long time and consistently provides satisfying experiences. The food is of high quality, and the staff is friendly.

**Step 2:**
- Long-standing reputation
- Consistent quality food
- Friendly staff


Topics for Good but something can be improved reviews:
Step 1: The restaurant has a long waiting line, but many customers find the food to be worth the wait. Customers compliment the delicious taste of the food, especially the pizzas and Halal dishes. It is mentioned that the cart offers generous portions 

1. what is the problem?

# Optional: Reformat Reviews

In [ ]:
def reformat_text(text):
    response = client.chat.completions.create(  # Use ChatCompletion instead of Completion
        model="gpt-3.5-turbo",  # Update to a supported model
        messages=[
            {"role": "system", "content": "You are a helpful assistant for reviews of restaurants."},
            {"role": "user", "content": f"what business processes are mentioned in this review?:\n\n{text}"}
        ],
        max_tokens=50
    )
    return response.choices[0].message.content.strip()  # Access content from message
# merged_df['summary'] = merged_df['Comment'].apply(summarize_text)

In [19]:
import pandas as pd
import glob

# List all CSV files
csv_files = glob.glob('*.csv')

# Initialize an empty list to store DataFrames
dataframes = []

# Loop through each CSV file and read it into a DataFrame
for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

# Concatenate all DataFrames
combined_df = pd.concat(dataframes, ignore_index=True)

# Drop duplicate rows
combined_df = combined_df.drop_duplicates(['Comment'])

# Save the resulting DataFrame to a new CSV file
combined_df.to_csv('sample_nyc_1000.csv', index=False)

In [26]:
combined_df[['Comment','topics']].sample(20)

,Comment,topics
447,"AMBIENCE: (4/5)\n- Interesting decor concept, ...",- Category: Ambiance\n - Description: The res...
433,This is my second time at Buddkan in Manhattan...,- Category: Ambiance\n - Description: The pla...
838,The best lobster rolls ever the service is gre...,- Category: Food Quality\n - Description: The...
716,My friends and I tried the margherita slice (b...,- Category: Food Quality\n - Description: Mar...
769,GEO at E14the is the King in this game of Pizz...,- Category: Food Quality\n - Description: GEO...
286,The Sicilian pie never disappoints! Baked ziti...,- Category: Food Quality\n - Description: The...
824,"Great food, fast and friendly service.\nFood w...",- Category: Food Quality\n - Description: Gre...
366,"We got appetizers instead of actual meals, and...",- Category: Food Quality\n - Description: Rec...
343,"Tried this place a couple of days ago, and it ...",- Category: Food Quality\n - Description: Ice...
974,Zero stars. This is not pizza this is mush. $5...,- Category: Food Quality\n - Description: Sog...


## Sentiment analysis

In [ ]:
from textblob import TextBlob

# Function to detect issues in comments
def detect_issues(comment):
    issues = []
    blob = TextBlob(comment)
    # Detect negative sentiment
    if blob.sentiment.polarity < 0:
        issues.append("Negative sentiment")
    # Detect specific keywords related to issues
    keywords = ['rude', 'poor', 'bad', 'slow', 'noisy', 'overpriced', 'disappointing', 'dirty', 'worst', 'not recommend']
    for keyword in keywords:
        if keyword in comment.lower():
            issues.append(keyword)
    return ', '.join(issues)

# Apply the function to the Comment column
reviews_df['Detected Issues'] = reviews_df['Comment'].apply(detect_issues)
